# 2 — Dataset & DataModule Objects

Prototype and validate the PyTorch `Dataset` and PyTorch Lightning `DataModule` objects before exporting them to `src/dataset.py`.

**`BrainDataset`**: loads pre-processed `.npy` spectrograms, applies `log1p` transform then per-fold standardisation (mean/std cached to JSON). Targets are soft labels: the vote distribution across 6 classes (sum to 1.0), which allows the KL divergence loss to leverage label uncertainty.

**`BrainDataModule`**: handles `StratifiedGroupKFold` splits (`n=5`, `seed=273`) grouped by `patient_id` to prevent patient leakage between train and validation sets. Split indices are cached to `data/cache/5_at_seed_273.json` for reproducibility. Ends with ad-hoc checks on null/NaN spectrograms.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
import numpy as np
from src.utils import *
from src.params import *
import json
from src.preprocess import *

import torch
from torch import nn
from torch.utils.data import Dataset, Subset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from pytorch_lightning import LightningDataModule, LightningModule

In [2]:
df = pd.read_csv(".." /DATA_DIR / "train.csv")

In [3]:
df

,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,1628180742,0,0.0,353733,0,0.0,127492639,42516,Seizure,3,0,0,0,0,0
1,1628180742,1,6.0,353733,1,6.0,3887563113,42516,Seizure,3,0,0,0,0,0
2,1628180742,2,8.0,353733,2,8.0,1142670488,42516,Seizure,3,0,0,0,0,0
3,1628180742,3,18.0,353733,3,18.0,2718991173,42516,Seizure,3,0,0,0,0,0
4,1628180742,4,24.0,353733,4,24.0,3080632009,42516,Seizure,3,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106795,351917269,6,12.0,2147388374,6,12.0,4195677307,10351,LRDA,0,0,0,3,0,0
106796,351917269,7,14.0,2147388374,7,14.0,290896675,10351,LRDA,0,0,0,3,0,0
106797,351917269,8,16.0,2147388374,8,16.0,461435451,10351,LRDA,0,0,0,3,0,0
106798,351917269,9,18.0,2147388374,9,18.0,3786213131,10351,LRDA,0,0,0,3,0,0


# Construction custom dataset

In [4]:
class BrainDataset(Dataset):

    def __init__(self, metadata, mean, std):

        super().__init__()
        self.metadata = metadata #metadata is the train.csv file
        self.mean = mean #mean and std stream-calculated in the datamodule over the train set
        self.std = std

    def len(self):
        return len(self.metadata)

    def __getitem__(self, idx):

        #load spec
        spec= load_spectrogram(df= self.metadata, idx= idx)

        #log transfo
        spec = np.log1p(spec) #should be before norm?

        #normalization
        spec = (spec - self.mean)/self.std # add epsilon to avoid break if self.std = 0?

        #conversion en tensor
        spec = torch.tensor(spec, dtype= torch.float32)

        #get votes
        votes = self.metadata.iloc[idx][VOTE_COL].values.astype(int)
        #print(votes)
        #print(votes.dtype)
        #convert into dist
        votes = votes/votes.sum()
        #convert into tensor
        votes = torch.from_numpy(votes).float()
        #print(votes.dtype)


        return spec, votes

In [5]:
class BrainDataModule(LightningDataModule):

    def __init__(self, metadata, batch_size= 32, num_workers= 4, seed= 273, n_split= 5, n_fold = 0):

        super().__init__()
        self.metadata = metadata # train.csv file, need again for split.
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.seed = seed
        self.n_split = n_split
        self.n_fold = n_fold


    def setup(self, stage= None):

        #temporary safety check
        if stage == "test":
            raise NotImplementedError("Test stage is not supported in BrainDataModule.")

        #check if split at given random state already cached

        cache_path = Path(CACHE_DIR / f"{self.n_split}_at_seed_{self.seed}.json")
        if cache_path.exists():

            print(f"[setup] Loading splits from cache: {cache_path}")
            with open(cache_path) as f:
                splits_dict = json.load(f)
            print(f"[setup] Cache loaded — {len(splits_dict)} folds found")

        else:
            #no cache data, need to do that split and calculate mean and std on train set
            print(f"[setup] No cache found — computing {self.n_split}-fold split (seed={self.seed})")

            sgkf = StratifiedGroupKFold(n_splits= self.n_split, random_state= self.seed, shuffle= True)
            groups = self.metadata["patient_id"]
            X = range(len(self.metadata))
            y = self.metadata["expert_consensus"]

            #dict to hold data on that split
            splits_dict = {}

            #iterate through splits
            for i, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):

                print(f"\n[setup] Fold {i+1}/{self.n_split} — {len(train_idx)} train / {len(val_idx)} val")
                splits_dict[str(i)] = {} #keys are fold number, values are dict with fold data
                splits_dict[str(i)]["train_idx"] = train_idx.tolist()
                splits_dict[str(i)]["val_idx"] = val_idx.tolist()

                #calcuate train set mean and std in a streaming fashion (approximation)
                mean = 0
                var = 0
                for idx in tqdm(train_idx, desc=f"  Fold {i+1} mean/std", unit="spec"):
                    spec = load_spectrogram(self.metadata, idx)
                    spec = np.log1p(spec)
                    mean += np.mean(spec)
                    var += np.var(spec)

                final_mean = mean/len(train_idx)
                final_std = np.sqrt(var/len(train_idx))
                print(f"  → mean={final_mean:.4f}, std={final_std:.4f}")

                splits_dict[str(i)]["mean"] = final_mean.tolist()
                splits_dict[str(i)]["std"] = final_std.tolist()

            #save json
            print(f"\n[setup] Saving cache to {cache_path}")
            with open(cache_path, "w") as f:
                json.dump(splits_dict, f)

            #get data for the desired fold
        self.train_idx = np.array(splits_dict[str(self.n_fold)]["train_idx"])
        self.val_idx = np.array(splits_dict[str(self.n_fold)]["val_idx"])
        self.mean = np.array(splits_dict[str(self.n_fold)]["mean"])
        self.std = np.array(splits_dict[str(self.n_fold)]["std"])

        print(f"train_idx dtype: {self.train_idx.dtype}")
        print(f"val_idx   dtype: {self.val_idx.dtype}")
        print(f"mean      dtype: {self.mean.dtype}")
        print(f"std       dtype: {self.std.dtype}")

        self.dataset = BrainDataset(metadata= self.metadata, mean= self.mean, std= self.std)


        if stage in ("fit", None):

            self.train_ds = Subset(self.dataset, self.train_idx)
            self.val_ds = Subset(self.dataset, self.val_idx)

    def train_dataloader(self):
        return DataLoader(self.train_ds,
                          batch_size= self.batch_size,
                          shuffle= True,
                          num_workers= self.num_workers,
                          pin_memory= True,
                          persistent_workers= self.num_workers > 0)


    def val_dataloader(self):
        return DataLoader(self.val_ds,
                          batch_size= self.batch_size,
                          shuffle= False,
                          num_workers= self.num_workers,
                          pin_memory= True,
                          persistent_workers= self.num_workers > 0)


In [6]:
test_dm = BrainDataModule(metadata= df)

In [7]:
test_dm.setup()

[setup] Loading splits from cache: /Users/pablorougerie/code/Projets/brain_waves/data/cache/5_at_seed_273.json
[setup] Cache loaded — 5 folds found
train_idx dtype: int64
val_idx   dtype: int64
mean      dtype: float64
std       dtype: float64


In [8]:
test_dm.mean

np.float64(1.0898381638150805)

In [9]:
test_dm.std

np.float64(1.3270619772709327)

In [10]:
test_dm.null

[]

In [11]:
spec_null = load_spectrogram(df, idx= 74)

In [12]:
df.iloc[74]

eeg_id                               374504640
eeg_sub_id                                   5
eeg_label_offset_seconds                  26.0
spectrogram_id                         3452193
spectrogram_sub_id                           5
spectrogram_label_offset_seconds          26.0
label_id                            2864386940
patient_id                               29847
expert_consensus                          GRDA
seizure_vote                                 0
lpd_vote                                     0
gpd_vote                                     2
lrda_vote                                    0
grda_vote                                    8
other_vote                                   2
Name: 74, dtype: object

In [13]:
raw_null = load_parquet(3452193)

In [14]:
raw_null.to_csv("null.csv")

In [15]:
eeg_null = pd.read_parquet(DATA_DIR / "train_eegs" /f"{374504640}.parquet")

In [16]:
eeg_null.isna().sum()

Fp1    0
F3     0
C3     0
P3     0
F7     0
T3     0
T5     0
O1     0
Fz     0
Cz     0
Pz     0
Fp2    0
F4     0
C4     0
P4     0
F8     0
T4     0
T6     0
O2     0
EKG    0
dtype: int64